In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"] =os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-3.1-flash")

### SUMMARIZATION MIDDLEWARE


In [ ]:
import langchain
langchain.debug = True

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("messages", 8),
            keep=("messages", 4),
        )
    ]
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "test1"
    }
}

In [ ]:
import langchain
langchain.debug = True
questions=[
    "what is the 2+2?",
    "what is the 2*2?",
    "what is the 2/2?",
    "what is the 2-2?",
]
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response['messages']}")
    print(f"Messages:{len(response['messages'])}")

#### 🔁 Middleware lifecycle
User input <br>
   ↓<br>
Add to memory <br>
   ↓<br>
Check message count<br>
   ↓<br>
If > 8 → summarize using LLM<br>
   ↓<br>
Replace old history<br>
   ↓<br>
Continue chat<br>


Other than this using message size we can also use Tokens count,fraction 

### HUMAN IN THE LOOP MIDDLEWARE

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID."""
    return f"Email {email_id} read."
def send_email_tool(reciept:str, subject:str ,body:str)->str:
    """Mock function to send an email by its ID."""
    return f"Email sent to {reciept} with subject'{subject}'"


In [72]:
agent  = create_agent(
    model = "google_genai:gemini-3.1-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                
                    "allowed_decisions":["approve","edit","reject"]
                },

                "read_email_tool":False,
            }
        )
    ]

)

In [73]:
from langchain_core.messages import HumanMessage,SystemMessage
config = {"configurable": {"thread_id": "test-approve"}}
result =  agent.invoke(
    {"messages":[HumanMessage(content="Send email to abc@gmail.com with subject 'Important reminderMiddleware(' and body 'Please remember to submit your report by Friday.'")]},
    config=config
)


In [74]:
result['messages'][0]

HumanMessage(content="Send email to abc@gmail.com with subject 'Important reminderMiddleware(' and body 'Please remember to submit your report by Friday.'", additional_kwargs={}, response_metadata={}, id='aa07e064-8ff6-4105-908a-2f47007a9ead')

In [75]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving...")
    # from langgraph.types import Command

    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}),
        config=config
    )
# print(result)

print(f"Result:{result['messages'][-1].content}")

Paused! Approving...
Result:[{'type': 'text', 'text': 'The request to send an email was rejected. If you would like me to proceed with a different action or need help with something else, please let me know.', 'extras': {'signature': 'EjQKMgEMOdbH6Kk8CIuhMYfhcLRC4cKKF19BcE4UrqTa4/NMj1p2/8/xJUXsgru9MbI4+W4v'}}]
